# 🚗 JalanCerdas AI - YOLO Training Notebook

Complete training pipeline for pothole detection model.

**Steps:**
1. Install dependencies
2. Download/prepare dataset
3. Visualize dataset samples
4. Configure training
5. Train model
6. Evaluate model
7. Export to TFLite
8. Test inference

## Cell 1: Install Dependencies

Install ultralytics, pyyaml, and other required packages.

In [ ]:
# Install required packages
!pip install -q ultralytics pyyaml tqdm Pillow matplotlib

import torch
from ultralytics import YOLO
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️  No GPU detected, will use CPU")

print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics installed ✓")

## Cell 2: Download/Prepare Dataset

Download pothole dataset from Kaggle or Roboflow, convert to YOLO format.

In [ ]:
# Option A: Run the download script
# !python scripts/download_dataset.py --source kaggle --output ../dataset

# Option B: Use local dataset (update path)
# !python scripts/download_dataset.py --source local --local-path /path/to/your/dataset

# Option C: Manual preparation
# Place your images in dataset/train/images/ and labels in dataset/train/labels/
# Same for val/ and test/ splits

# Verify dataset structure
dataset_path = Path("../dataset")
data_yaml = dataset_path / "data.yaml"

if data_yaml.exists():
    with open(data_yaml) as f:
        config = yaml.safe_load(f)
    print(f"Dataset config: {data_yaml}")
    print(f"Classes: {config['nc']} - {config['names']}")
    
    for split in ['train', 'val', 'test']:
        img_dir = dataset_path / config.get(split, f'{split}/images')
        if img_dir.exists():
            n = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
            print(f"  {split}: {n} images")
        else:
            print(f"  {split}: not found")
else:
    print(f"⚠️  No data.yaml found at {data_yaml}")
    print("Please run download_dataset.py first, or place dataset manually.")
    print("\nDataset structure needed:")
    print("  dataset/")
    print("  ├── train/images/*.jpg")
    print("  ├── train/labels/*.txt")
    print("  ├── val/images/*.jpg")
    print("  ├── val/labels/*.txt")
    print("  ├── test/images/*.jpg")
    print("  ├── test/labels/*.txt")
    print("  └── data.yaml")

## Cell 3: Visualize Dataset Samples

Display sample images with their bounding box annotations.

In [ ]:
import random
from PIL import Image, ImageDraw, ImageFont

def show_samples(dataset_path, split='train', n_samples=6):
    """Show sample images with annotations."""
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    
    if not img_dir.exists():
        print(f"Directory not found: {img_dir}")
        return
    
    # Get images with labels
    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    if not images:
        print("No images found")
        return
    
    samples = random.sample(images, min(n_samples, len(images)))
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, img_path in enumerate(samples):
        if i >= len(axes):
            break
            
        img = Image.open(img_path)
        draw = ImageDraw.Draw(img)
        w, h = img.size
        
        # Load labels
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls, cx, cy, bw, bh = int(parts[0]), *[float(x) for x in parts[1:5]]
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        x2 = int((cx + bw/2) * w)
                        y2 = int((cy + bh/2) * h)
                        draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
                        draw.text((x1, y1-15), 'pothole', fill='red')
        
        axes[i].imshow(img)
        axes[i].set_title(img_path.name, fontsize=8)
        axes[i].axis('off')
    
    plt.suptitle(f"Sample {split} images with annotations", fontsize=14)
    plt.tight_layout()
    plt.show()

if data_yaml.exists():
    show_samples(dataset_path, 'train', 6)
else:
    print("Dataset not prepared yet. Run Cell 2 first.")

## Cell 4: Configure Training

Set up training hyperparameters and model selection.

In [ ]:
# === Training Configuration ===

CONFIG = {
    # Model
    'model': 'yolov8n.pt',     # Options: yolov8n.pt, yolov8s.pt, yolo11n.pt, yolo11s.pt
    
    # Dataset
    'data': '../dataset/data.yaml',
    
    # Training
    'epochs': 100,             # Number of epochs
    'batch': 16,               # Batch size (reduce if OOM)
    'imgsz': 640,              # Input image size
    'patience': 50,            # Early stopping patience
    
    # Optimizer
    'optimizer': 'auto',       # SGD, Adam, AdamW, auto
    'lr0': 0.01,               # Initial learning rate
    'lrf': 0.01,               # Final learning rate factor
    'momentum': 0.937,
    'weight_decay': 0.0005,
    
    # Augmentation
    'mosaic': 1.0,             # Mosaic probability
    'hsv_h': 0.015,            # HSV-Hue
    'hsv_s': 0.7,              # HSV-Saturation
    'hsv_v': 0.4,              # HSV-Value
    
    # Output
    'project': 'runs',
    'name': 'train_notebook',
}

device = '0' if torch.cuda.is_available() else 'cpu'
CONFIG['device'] = device

print("Training Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Cell 5: Train Model

Start YOLO training with configured parameters.

In [ ]:
import time

# Load model
model = YOLO(CONFIG['model'])

# Start training
start_time = time.time()
print("🚀 Starting training...")

results = model.train(
    data=CONFIG['data'],
    epochs=CONFIG['epochs'],
    batch=CONFIG['batch'],
    imgsz=CONFIG['imgsz'],
    device=CONFIG['device'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    patience=CONFIG['patience'],
    mosaic=CONFIG['mosaic'],
    hsv_h=CONFIG['hsv_h'],
    hsv_s=CONFIG['hsv_s'],
    hsv_v=CONFIG['hsv_v'],
    project=CONFIG['project'],
    name=CONFIG['name'],
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start_time
hours, rem = divmod(elapsed, 3600)
mins, secs = divmod(rem, 60)

print(f"\n✅ Training complete in {int(hours)}h {int(mins)}m {int(secs)}s")
print(f"Results saved to: {results.save_dir}")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## Cell 6: Evaluate Model

Run validation and display key metrics.

In [ ]:
from pathlib import Path

# Find best weights
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'

if best_weights.exists():
    # Validate
    val_model = YOLO(str(best_weights))
    val_results = val_model.val(data=CONFIG['data'], device=CONFIG['device'])
    
    # Display metrics
    print("\n📊 Validation Results:")
    print(f"  mAP@50:     {val_results.box.map50:.4f}")
    print(f"  mAP@50-95:  {val_results.box.map:.4f}")
    print(f"  Precision:  {val_results.box.mp:.4f}")
    print(f"  Recall:     {val_results.box.mr:.4f}")
    
    f1 = 2 * (val_results.box.mp * val_results.box.mr) / (val_results.box.mp + val_results.box.mr + 1e-8)
    print(f"  F1 Score:   {f1:.4f}")
    
    # Plot confusion matrix
    fig_dir = Path(results.save_dir)
    confusion_file = fig_dir / 'confusion_matrix.png'
    if confusion_file.exists():
        img = Image.open(confusion_file)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.title('Confusion Matrix')
        plt.axis('off')
        plt.show()
else:
    print(f"⚠️  Best weights not found at {best_weights}")

## Cell 7: Export to TFLite

Export trained model to TFLite format for mobile deployment.

In [ ]:
if best_weights.exists():
    # Load trained model
    export_model = YOLO(str(best_weights))
    
    # Export to TFLite (FP16)
    print("📦 Exporting to TFLite...")
    export_path = export_model.export(
        format='tflite',
        imgsz=CONFIG['imgsz'],
        half=True,          # FP16 quantization
        simplify=True,
    )
    
    export_file = Path(export_path)
    size_mb = export_file.stat().st_size / (1024 * 1024) if export_file.exists() else 0
    
    print(f"\n✅ Export complete!")
    print(f"  Model: {export_path}")
    print(f"  Size:  {size_mb:.2f} MB")
    
    if size_mb > 10:
        print(f"  ⚠️  Model exceeds 10MB target ({size_mb:.2f} MB)")
        print("  Consider using --int8 for smaller model")
else:
    print("⚠️  No trained weights found. Run training first.")

## Cell 8: Test Inference

Run inference on sample images to verify model works correctly.

In [ ]:
from pathlib import Path
import random

if best_weights.exists():
    # Load model
    infer_model = YOLO(str(best_weights))
    
    # Get sample images from test or val set
    test_images = list(Path('../dataset/test/images').glob('*.jpg')) if Path('../dataset/test/images').exists() else []
    val_images = list(Path('../dataset/val/images').glob('*.jpg')) if Path('../dataset/val/images').exists() else []
    all_images = test_images or val_images
    
    if not all_images:
        print("⚠️  No test/val images found for inference demo")
    else:
        # Run inference on 6 random samples
        samples = random.sample(all_images, min(6, len(all_images)))
        
        results_list = infer_model.predict(
            source=[str(s) for s in samples],
            conf=0.25,
            device=CONFIG['device'],
            save=True,
            project='runs',
            name='predict_demo',
            exist_ok=True,
        )
        
        # Show results
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, (result, sample) in enumerate(zip(results_list, samples)):
            if i >= len(axes):
                break
            
            # Plot with bounding boxes
            plotted = result.plot()  # Returns numpy array with boxes drawn
            axes[i].imshow(plotted[..., ::-1])  # BGR to RGB
            axes[i].set_title(sample.name, fontsize=8)
            axes[i].axis('off')
            
            # Print detections
            for box in result.boxes:
                conf = float(box.conf)
                cls = int(box.cls)
                label = result.names[cls]
                print(f"  {sample.name}: {label} ({conf:.2%})")
        
        plt.suptitle("Pothole Detection Results", fontsize=14)
        plt.tight_layout()
        plt.show()
        
        print(f"\n✅ Inference complete. Results saved to runs/predict_demo/")
else:
    print("⚠️  No trained weights found. Run training first.")

## Summary

**Pipeline complete!** Next steps:

1. Copy TFLite model to `mobile/assets/models/`
2. Add model to Flutter `pubspec.yaml` assets
3. Load model in Flutter with `tflite_flutter` package
4. Run inference on camera frames

For production training, use `scripts/train.py` for better logging and reproducibility.